### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="heart_disease_switzerland",
    dataset_year="1989",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C52P4X",
    download_description="""
Get the UCI data.

wget https://archive.ics.uci.edu/static/public/45/heart+disease.zip && unzip heart+disease.zip processed.switzerland.data && rm heart+disease.zip && mkdir -p local-data-warehouse/heart_disease_switzerland && mv processed.switzerland.data local-data-warehouse/heart_disease_switzerland/
""",
    # References
    academic_reference_bibtex="""@article{detrano1989international,
  title={International application of a new probability algorithm for the diagnosis of coronary artery disease},
  author={Detrano, Robert and Janosi, Andras and Steinbrunn, Walter and Pfisterer, Matthias and Schmid, Johann-Jakob and Sandhu, Sarbjit and Guppy, Kern H and Lee, Stella and Froelicher, Victor},
  journal={The American journal of cardiology},
  volume={64},
  number={5},
  pages={304--310},
  year={1989},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="detrano1989international",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the processed version and the subset of 14 attributes used in the study and clinical practice.

- We encode missing values as np.nan instead of "?".
- We make the target binary (0=no heart disease, 1=heart disease). This follows the original study in attempting to distinguish presence (values 1,2,3,4) from absence (value 0).
- We drop the "chol" feature, which is constant in this data.
- The dataset has only 8 patients without a heart disease diagnosis, which is a very small sample size for the negative class. This likely makes the task trivial.
- There is also a negative value for "oldpeak", which might be a data error as all other datasets only have positive values for this value.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="heart_disease_diagnosis",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="heart_disease_diagnosis",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "num"
]

df = pd.read_csv(dataset_mold.path / "processed.switzerland.data", header=None, names=columns)
print("Loaded data shape:", df.shape)

df = df.replace("?", np.nan)
df[["ca", "trestbps", "thalach", "oldpeak"]] = df[["ca", "trestbps", "thalach", "oldpeak"]].astype(float)
# Make target
df["heart_disease_diagnosis"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])

as_cat_type = ["thal", "slope", "exang", "restecg", "fbs", "cp", "sex", "heart_disease_diagnosis"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=["chol"]) # Constant

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (123, 14)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 123
Columns: 13
Use sampling: False (sample size: 123)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['thalach', 'age', 'oldpeak', 'trestbps', 'cp', 'thal', 'slope', 'restecg', 'fbs', 'exang']
Rows remaining as candidates after top-10 filter: 0 (of 123)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,sex,cp,trestbps,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,heart_disease_diagnosis
0,46,1,4,100.0,NaN,1,133.0,0,-2.6,2,NaN,NaN,1
1,53,1,4,125.0,NaN,0,120.0,0,1.5,1,NaN,NaN,1
2,53,1,4,80.0,NaN,0,141.0,1,2.0,3,NaN,NaN,0
3,61,1,4,150.0,0,0,105.0,1,0.0,2,NaN,7,1
4,38,0,4,105.0,NaN,0,166.0,0,2.8,1,NaN,NaN,1


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,fbs,category,75.0,60.98,2.0,"0, 1"
1,thal,category,52.0,42.28,3.0,"7, 3, 6"
2,slope,category,17.0,13.82,3.0,"2, 1, 3"
3,restecg,category,1.0,0.81,3.0,"0, 1, 2"
4,exang,category,1.0,0.81,2.0,"0, 1"
5,sex,category,0.0,0.00,2.0,"1, 0"
6,cp,category,0.0,0.00,4.0,"4, 3, 1, 2"
7,heart_disease_diagnosis,category,0.0,0.00,2.0,"1, 0"
8,ca,float64,118.0,95.93,2.0,"2.0, 1.0"
9,oldpeak,float64,6.0,4.88,35.0,"0.0, 2.0, 1.0, 1.5, 0.5, 0.7, 0.1, 2.5, 1.4, -1.0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,123.0,55.317073,9.032108,32.0,74.0
trestbps,121.0,130.206612,22.559151,80.0,200.0
thalach,122.0,121.557377,25.977438,60.0,182.0
oldpeak,117.0,0.653846,1.056061,-2.6,3.7
ca,5.0,1.600000,0.547723,1.0,2.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                  rank                    
cp                      1        4     98  79.67
                        2        3     17  13.82
                        3        1      4   3.25
                        4        2      4   3.25
exang                   1        0     68  55.28
                        2        1     54  43.90
                        3     <NA>      1   0.81
fbs                     1     <NA>     75  60.98
                        2        0     43  34.96
                        3        1      5   4.07
heart_disease_diagnosis 1        1    115  93.50
                        2        0      8   6.50
restecg                 1        0     85  69.11
                        2        1     30  24.39
                        3        2      7   5.69
                        4     <NA>      1   0.81
sex                     1        1    113  91.87
                        2        0     10   8.13
slope                   1        2     61  49.59
                        2        1     33  26.83
                        3     <NA>     17  13.82
                        4        3     12   9.76
thal                    1     <NA>     52  42.28
                        2        7     42  34.15
                        3        3     19  15.45
                        4        6     10   8.13

In [8]:
# Target Distribution
target_df

,count,pct
heart_disease_diagnosis,,
1,115,93.5
0,8,6.5


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to heart_disease_switzerland/019d9cdf-3ce5-7e78-840a-3dffae2b965e
019d9cdf-3ce5-7e78-840a-3dffae2b965e
7a4228ad3c2af65cf035bf995869492c959275b29b66d0f043d3616c6ef572cf
